## 0. Load Packages

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import sys
sys.executable, sys.version


('/Users/jisoyun/Desktop/sparta_python/ Logistics/.venv/bin/python',
 '3.9.6 (default, Feb  3 2024, 15:58:27) \n[Clang 15.0.0 (clang-1500.3.9.4)]')

### State(주)별 물류 허브 파워 분석

사용 변수  
- dms_orig, dms_dest  
- tons_YYYY, value_YYYY  

파생 변수  
- Outbound Tons (출발 물동량)  
- Inbound Tons (도착 물동량)  
- Hub Index = Outbound + Inbound  
- Value per Ton = value / tons  

분석 질문  
- 가장 많은 화물을 보내는 주는 어디인가?  
- 가장 많은 화물을 받는 주는 어디인가?  
- 물량 대비 가치가 높은 주는 어디인가?  

---
- (1) 연도 컬럼 long-format
        ↓
- (2) Domestic만 필터
        ↓
- (3) State 단위 집계
        ↓
- (4) 파생변수 생성
        ↓
- (5) 랭킹 테이블 생성
        ↓
- (6) 시각화

#### 기존 데이터 국내 / 수입 / 수출로 분리 및 EDA를 위한 메타 데이터 결합

메타 데이터 결합을 통해 코드를 그에 맞는 이름으로 변환하고, _nm을 붙여 시각화에 용이하게끔 데이터 재구성

In [5]:
df = pd.read_parquet("../data/FAF5_join_meta.parquet")

In [11]:
df.dtypes

fr_orig                float64
dms_orig                 int64
dms_dest                 int64
fr_dest                float64
fr_inmode              float64
dms_mode                 int64
fr_outmode             float64
sctg2                    int64
trade_type               int64
dist_band                int64
tons_2018              float64
tons_2019              float64
tons_2020              float64
tons_2021              float64
tons_2022              float64
tons_2023              float64
tons_2024              float64
value_2018             float64
value_2019             float64
value_2020             float64
value_2021             float64
value_2022             float64
value_2023             float64
value_2024             float64
current_value_2018     float64
current_value_2019     float64
current_value_2020     float64
current_value_2021     float64
current_value_2022     float64
current_value_2023     float64
current_value_2024     float64
tmiles_2018            float64
tmiles_2

In [6]:
df[['state_orig_nm','state_dest_nm']].head()


,state_orig_nm,state_dest_nm
0,Alabama,Alabama
1,Alabama,Alabama
2,Alabama,Florida
3,Alabama,Georgia
4,Alabama,Georgia


In [12]:
years = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

result = []

for y in years:

    outbound = (
        df.groupby("state_orig_nm",observed=True)[f"tons_{y}"]
          .sum()
          .reset_index()
          .rename(columns={
              "state_orig_nm": "state",
              f"tons_{y}": "outbound_tons"
          })
    )

    inbound = (
        df.groupby("state_dest_nm",observed=True)[f"tons_{y}"]
          .sum()
          .reset_index()
          .rename(columns={
              "state_dest_nm": "state",
              f"tons_{y}": "inbound_tons"
          })
    )

    hub = outbound.merge(inbound, on="state", how="outer")
    hub["year"] = y

    result.append(hub)

hub_long = pd.concat(result, ignore_index=True)

# Hub Index 계산
hub_long["hub_index"] = (
    hub_long["outbound_tons"].fillna(0) +
    hub_long["inbound_tons"].fillna(0)
)

In [13]:
hub_long["hub_index"] = (
    hub_long["outbound_tons"].fillna(0) +
    hub_long["inbound_tons"].fillna(0)
)



In [14]:
display(hub_long.head())

,state,outbound_tons,inbound_tons,year,hub_index
0,Alabama,4.102201e+05,4.194239e+05,2018,8.296440e+05
1,Alaska,5.470390e+04,3.456562e+04,2018,8.926952e+04
2,Arizona,1.898884e+05,2.269534e+05,2018,4.168418e+05
3,Arkansas,2.436029e+05,2.432365e+05,2018,4.868394e+05
4,California,1.229196e+06,1.315320e+06,2018,2.544516e+06


In [15]:
years = [2018, 2019, 2020, 2021, 2022, 2023, 2024]
result = []

for y in years:
    value_state = (
        df.groupby("state_orig_nm",observed=True)[[f"value_{y}", f"tons_{y}"]]
          .sum()
          .reset_index()
          .rename(columns={
              "state_orig_nm": "state",
              f"value_{y}": "value",
              f"tons_{y}": "tons"
          })
    )

    # value_per_ton 계산 (0으로 나누기 방지)
    value_state["value_per_ton"] = np.where(
        value_state["tons"] > 0,
        value_state["value"] / value_state["tons"],
        np.nan
    )

    value_state["year"] = y
    result.append(value_state)

# 최종 long format 데이터프레임
value_state_long = pd.concat(result, ignore_index=True)

value_state_long



,state,value,tons,value_per_ton,year
0,Alabama,2.883488e+05,4.102201e+05,0.702912,2018
1,Alaska,4.945820e+04,5.470390e+04,0.904107,2018
2,Arizona,2.287623e+05,1.898884e+05,1.204720,2018
3,Arkansas,1.478820e+05,2.436029e+05,0.607062,2018
4,California,2.342587e+06,1.229196e+06,1.905788,2018
...,...,...,...,...,...
352,Washington,4.298103e+05,4.589871e+05,0.936432,2024
353,Washington DC,1.215654e+04,6.357920e+03,1.912031,2024
354,West Virginia,7.103532e+04,2.351663e+05,0.302064,2024
355,Wisconsin,3.500936e+05,4.488725e+05,0.779940,2024


In [1]:
import sys
from utils import get_top_by_metric

# 하나의 함수로 모든 메트릭 처리
density_top = get_top_by_metric(df, 'value_per_ton', 2024, 10, clean_invalid=True)
value_top = get_top_by_metric(df, 'value', 2024, 10)
tons_top = get_top_by_metric(df, 'tons', 2024, 10)

ModuleNotFoundError: No module named 'utils'